In [2]:
import numpy as np
import pandas as pd

db = pd.read_csv("train_transaction.csv")
identity = pd.read_csv("train_identity.csv")

print("Transaction:", db.shape)
print("Identity:", identity.shape)

Transaction: (590540, 394)
Identity: (144233, 41)


In [3]:
data = db.merge(
    identity,
    on="TransactionID",
    how="left"
)

print("Merged:", data.shape)

Merged: (590540, 434)


In [ ]:
data = data.sort_values(
    "TransactionDT"
).reset_index(drop=True)

In [1]:
data["has_identity"] = (
    data["id_01"].notna()
).astype(np.int8)

NameError: name 'data' is not defined

In [17]:
data["log_TransactionAmt"] = np.log1p(
    data["TransactionAmt"]
)

In [18]:
data["TransactionHour"] = (
    (data["TransactionDT"] // 3600) % 24
).astype(np.int8)

In [19]:
data["TransactionDay"] = (
    data["TransactionDT"] // (24 * 3600)
).astype(np.int32)

In [20]:
data["time_since_previous"] = (
    data["TransactionDT"].diff()
).fillna(0)

In [21]:
data["amount_change"] = (
    data["TransactionAmt"].diff()
).fillna(0)

In [22]:
data["abs_amount_change"] = (
    data["amount_change"].abs()
)

In [23]:
data["card1_frequency"] = (
    data.groupby("card1").cumcount()
)

In [25]:
data["card1_previous_amount_sum"] = (
    data.groupby("card1")["TransactionAmt"]
    .cumsum()
    - data["TransactionAmt"]
)

In [26]:
data["card1_previous_count"] = (
    data.groupby("card1").cumcount()
)

In [27]:
data["card1_avg_previous_amount"] = (
    data["card1_previous_amount_sum"]
    /
    data["card1_previous_count"].replace(
        0,
        np.nan
    )
)

In [28]:
data["amount_vs_card_avg"] = (
    data["TransactionAmt"]
    /
    (data["card1_avg_previous_amount"] + 1e-6)
)

In [29]:
data["card1_avg_previous_amount"] = (
    data["card1_avg_previous_amount"]
    .fillna(data["TransactionAmt"])
)

In [30]:
data["amount_vs_card_avg"] = (
    data["TransactionAmt"]
    /
    (data["card1_avg_previous_amount"] + 1e-6)
)

In [31]:
data["recent_card_transactions"] = (
    data.groupby("card1")["TransactionDT"]
    .transform(
        lambda x: x.rolling(
            window=10,
            min_periods=1
        ).count()
    )
)

In [32]:
data = data.drop(
    columns=[
        "card1_previous_amount_sum",
        "card1_previous_count"
    ]
)

In [33]:
behavioral_features = [
    "log_TransactionAmt",
    "TransactionHour",
    "TransactionDay",
    "has_identity",
    "time_since_previous",
    "amount_change",
    "abs_amount_change",
    "card1_frequency",
    "card1_avg_previous_amount",
    "amount_vs_card_avg",
    "recent_card_transactions"
]

print("Behavioral features:")
print(behavioral_features)

Behavioral features:
['log_TransactionAmt', 'TransactionHour', 'TransactionDay', 'has_identity', 'time_since_previous', 'amount_change', 'abs_amount_change', 'card1_frequency', 'card1_avg_previous_amount', 'amount_vs_card_avg', 'recent_card_transactions']


In [34]:
data[behavioral_features].describe().T

,count,mean,std,min,25%,50%,75%,max
log_TransactionAmt,160094.0,4.375635,0.910616,0.256191,3.806662,4.330733,4.836282,8.536201
TransactionHour,160094.0,13.840625,7.645866,0.000000,6.000000,16.000000,20.000000,23.000000
TransactionDay,160094.0,18.991667,10.241028,1.000000,11.000000,19.000000,27.000000,38.000000
has_identity,160094.0,0.396879,0.489252,0.000000,0.000000,0.000000,1.000000,1.000000
time_since_previous,160094.0,20.477132,46.782377,0.000000,4.000000,10.000000,22.000000,3490.000000
amount_change,160094.0,-0.000067,281.318041,-5001.950000,-60.000000,0.000000,60.000000,4944.950000
abs_amount_change,160094.0,134.146872,247.273825,0.000000,24.067000,60.000000,142.050000,5001.950000
card1_frequency,160094.0,360.782884,622.286288,0.000000,11.000000,85.000000,399.000000,3967.000000
card1_avg_previous_amount,160094.0,129.706610,101.634888,1.896000,81.146677,113.711519,156.250000,3427.770000
amount_vs_card_avg,160094.0,1.089314,1.723584,0.005281,0.408101,0.726709,1.149959,136.639324


In [35]:
print(
    "Missing values:",
    data[behavioral_features].isnull().sum().sum()
)

Missing values: 0


In [36]:
print(
    "Infinite values:",
    np.isinf(
        data[behavioral_features]
        .select_dtypes(include=np.number)
    ).sum().sum()
)

Infinite values: 0


In [37]:
import os
os.makedirs(
    "../data/processed",
    exist_ok=True
)
data.to_parquet(
    "../data/processed/razorshield_engineered_590k.parquet",
    index=False
)
print("Saved successfully!")
print("Shape:", data.shape)

Saved successfully!
Shape: (160094, 445)


In [38]:
data.groupby("isFraud")[
    "amount_vs_card_avg"
].agg(
    ["count", "mean", "median"]
)

,count,mean,median
isFraud,,,
0,155724,1.083901,0.721983
1,4370,1.282211,0.912555


In [39]:
data.groupby("isFraud")[
    "card1_frequency"
].agg(
    ["count", "mean", "median"]
)

,count,mean,median
isFraud,,,
0,155724,360.057987,85.0
1,4370,386.614416,103.5


In [40]:
data.groupby("has_identity")[
    "isFraud"
].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
has_identity,,,
0,96556,1878,0.019450
1,63538,2492,0.039221


In [41]:
data.groupby("TransactionHour")[
    "isFraud"
].agg(
    ["count", "sum", "mean"]
).sort_values(
    "mean",
    ascending=False
)

,count,sum,mean
TransactionHour,,,
9,604,52,0.086093
8,782,46,0.058824
6,1990,110,0.055276
7,1205,64,0.053112
5,3211,166,0.051697
4,4500,220,0.048889
10,692,33,0.047688
23,10442,329,0.031507
3,5839,180,0.030827


In [1]:
import numpy as np
import pandas as pd

data = pd.read_parquet(
    "../data/processed/razorshield_engineered_590k.parquet"
)

print("Data shape:", data.shape)

Data shape: (160094, 445)


In [2]:
data = data.sort_values(
    "TransactionDT"
).reset_index(drop=True)

In [3]:
X = data.drop(
    columns=["isFraud", "TransactionID"]
)

y = data["isFraud"]

print("X:", X.shape)
print("y:", y.shape)

X: (160094, 443)
y: (160094,)


In [4]:
split_1 = int(len(X) * 0.70)
split_2 = int(len(X) * 0.85)

X_train = X.iloc[:split_1].copy()
X_val = X.iloc[split_1:split_2].copy()
X_test = X.iloc[split_2:].copy()

y_train = y.iloc[:split_1].copy()
y_val = y.iloc[split_1:split_2].copy()
y_test = y.iloc[split_2:].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (112065, 443)
Validation: (24014, 443)
Test: (24015, 443)


In [5]:
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())
print("Test fraud rate:", y_test.mean())

print("\nFraud counts:")
print("Train:", y_train.sum())
print("Validation:", y_val.sum())
print("Test:", y_test.sum())

Train fraud rate: 0.024360861999732298
Validation fraud rate: 0.0324810527192471
Test fraud rate: 0.035810951488652924

Fraud counts:
Train: 2730
Validation: 780
Test: 860


In [6]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns:", len(categorical_columns))
print(categorical_columns)

Categorical columns: 31
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [7]:
sparse_columns = [
    col
    for col in X_train.columns
    if X_train[col].isnull().mean() > 0.90
]

print("Sparse columns:", len(sparse_columns))
print(sparse_columns)

Sparse columns: 12
['dist2', 'D7', 'D13', 'id_07', 'id_08', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']


In [8]:
X_train = X_train.drop(columns=sparse_columns)
X_val = X_val.drop(columns=sparse_columns)
X_test = X_test.drop(columns=sparse_columns)

In [9]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=["int64", "float64", "float32", "int32"]
).columns.tolist()

print("Categorical:", len(categorical_columns))
print("Numerical:", len(numerical_columns))

Categorical: 29
Numerical: 400


In [10]:
train_medians = X_train[numerical_columns].median()

X_train[numerical_columns] = X_train[numerical_columns].fillna(
    train_medians
)

X_val[numerical_columns] = X_val[numerical_columns].fillna(
    train_medians
)

X_test[numerical_columns] = X_test[numerical_columns].fillna(
    train_medians
)

In [11]:
X_train[categorical_columns] = (
    X_train[categorical_columns].fillna("Unknown")
)

X_val[categorical_columns] = (
    X_val[categorical_columns].fillna("Unknown")
)

X_test[categorical_columns] = (
    X_test[categorical_columns].fillna("Unknown")
)

In [12]:
high_cardinality_columns = [
    col
    for col in categorical_columns
    if X_train[col].nunique() > 100
]

print(
    "High-cardinality columns:",
    high_cardinality_columns
)

High-cardinality columns: ['id_33', 'DeviceInfo']


In [13]:
X_train = X_train.drop(
    columns=high_cardinality_columns
)

X_val = X_val.drop(
    columns=high_cardinality_columns
)

X_test = X_test.drop(
    columns=high_cardinality_columns
)

In [14]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns
)

X_val = pd.get_dummies(
    X_val,
    columns=categorical_columns
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns
)

In [15]:
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [16]:
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (112065, 753)
Validation: (24014, 753)
Test: (24015, 753)


In [17]:
print("Missing values:")

print(
    "Train:",
    X_train.isnull().sum().sum()
)

print(
    "Validation:",
    X_val.isnull().sum().sum()
)

print(
    "Test:",
    X_test.isnull().sum().sum()
)

Missing values:
Train: 0
Validation: 0
Test: 0


In [18]:
from xgboost import XGBClassifier

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(
    "Scale positive weight:",
    scale_pos_weight
)

Scale positive weight: 40.04945054945055


In [19]:
xgb_v2 = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    objective="binary:logistic",
    eval_metric="aucpr",

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)

In [20]:
xgb_v2.fit(
    X_train,
    y_train
)
print("XGBoost V2 training completed.")

XGBoost V2 training completed.


In [21]:
xgb_v2_val_prob = xgb_v2.predict_proba(
    X_val
)[:, 1]

In [22]:
xgb_v2_val_pred = (
    xgb_v2_val_prob >= 0.5
).astype(int)

In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

def evaluate_model(
    name,
    y_true,
    y_pred,
    y_prob
):

    print("=" * 60)
    print(name)
    print("=" * 60)

    print(
        f"Accuracy : {accuracy_score(y_true, y_pred):.4f}"
    )

    print(
        f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"F1       : {f1_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"ROC-AUC  : {roc_auc_score(y_true, y_prob):.4f}"
    )

    print(
        f"PR-AUC   : {average_precision_score(y_true, y_prob):.4f}"
    )

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )

In [24]:
evaluate_model(
    "RazorShield XGBoost V2 - Validation",
    y_val,
    xgb_v2_val_pred,
    xgb_v2_val_prob
)

RazorShield XGBoost V2 - Validation
Accuracy : 0.9718
Precision: 0.5855
Recall   : 0.4564
F1       : 0.5130
ROC-AUC  : 0.8853
PR-AUC   : 0.5166

Confusion Matrix:
[[22982   252]
 [  424   356]]


In [25]:
threshold_results = []

for threshold in np.arange(
    0.05,
    0.96,
    0.05
):

    pred = (
        xgb_v2_val_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_val,
            pred,
            zero_division=0
        )
    })

threshold_df_v2 = pd.DataFrame(
    threshold_results
)

threshold_df_v2

,threshold,precision,recall,f1
0,0.05,0.082069,0.880769,0.150148
1,0.10,0.128502,0.782051,0.220735
2,0.15,0.177699,0.708974,0.284173
3,0.20,0.231423,0.662821,0.343066
4,0.25,0.291415,0.617949,0.396056
5,0.30,0.346780,0.573077,0.432093
6,0.35,0.418057,0.546154,0.473596
7,0.40,0.476998,0.505128,0.490660
8,0.45,0.532764,0.479487,0.504723
9,0.50,0.585526,0.456410,0.512968


In [26]:
best_row_v2 = threshold_df_v2.loc[
    threshold_df_v2["f1"].idxmax()
]

best_threshold_v2 = best_row_v2["threshold"]

print("Best threshold:", best_threshold_v2)
print("Precision:", best_row_v2["precision"])
print("Recall:", best_row_v2["recall"])
print("F1:", best_row_v2["f1"])

Best threshold: 0.5
Precision: 0.5855263157894737
Recall: 0.4564102564102564
F1: 0.5129682997118156


In [27]:
xgb_v2_test_prob = xgb_v2.predict_proba(
    X_test
)[:, 1]

xgb_v2_test_pred = (
    xgb_v2_test_prob >= best_threshold_v2
).astype(int)

In [28]:
evaluate_model(
    "RazorShield XGBoost V2 - FINAL TEST",
    y_test,
    xgb_v2_test_pred,
    xgb_v2_test_prob
)

RazorShield XGBoost V2 - FINAL TEST
Accuracy : 0.9692
Precision: 0.5921
Recall   : 0.4523
F1       : 0.5129
ROC-AUC  : 0.8668
PR-AUC   : 0.5147

Confusion Matrix:
[[22887   268]
 [  471   389]]


In [32]:
!pip install shap -q

In [29]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [30]:
print("Model:", type(xgb_v2))

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Model: <class 'xgboost.sklearn.XGBClassifier'>
X_train: (112065, 753)
X_val: (24014, 753)
X_test: (24015, 753)
